# Suite PAT — Patterns: symbols, variables, constraints, locators, expressions and collectors

A pattern (`Schemas.OfPattern`) is the shape of a frame ([FEA-027: mutations as queries](../../../mbse-basis/docs/features/FEA-027.md)): its symbols, each standing for a
reference object, its variables, each standing for a value, and the constraints a frame meets, expressions of boolean
result.
An expression (`Schemas.OfExpression`) ranges over a pattern; its terms are mbse-expressions'. A collector
(`Schemas.OfCollector`) folds a pattern's frames. Each is a schema object, built, validated and written as every one is;
frames, and matching, are the executor's (step 4e of the store overhaul).

In [ ]:
import { Builders as B, JSON as Json, Plain, Schemas as S } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

/** wts.Features, a singleton directory owning definitions through `dir`, keyed by name, each owning its clauses
 * through `clauses`. */
function wts(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("wts/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Directory: S.OfObject, Definition: S.OfObject, Clause: S.OfObject, Defines: S.OfRelation,
      HasClause: S.OfRelation })
    .spec([
      (_: any) => _.Directory.name("wts.Directory").ref(true).singleton("wts.Features").adjacencies([
        (t: any) => t.name("dir").relation([_.Defines]).me("directory")]),
      (_: any) => _.Definition.name("wts.Definition").ref(true).properties([
        (t: any) => t.name("title").schema([S.OfStr]),
      ]).adjacencies([
        (t: any) => t.name("def").relation([_.Defines]).me("defn").owner(true),
        (t: any) => t.name("clauses").relation([_.HasClause]).me("defn"),
      ]),
      (_: any) => _.Clause.name("wts.Clause").ref(true).adjacencies([
        (t: any) => t.relation([_.HasClause]).me("clause").owner(true)]),
      (_: any) => _.Defines.name("wts.Defines").links([(t: any) => t.name("directory"), (t: any) => t.name("defn")]).properties([
        (t: any) => t.name("name").schema([S.OfStr])]).uniques([["directory", "name"], ["defn"]]),
      (_: any) => _.HasClause.name("wts.HasClause").links([(t: any) => t.name("defn"), (t: any) => t.name("clause")]).properties([
        (t: any) => t.name("name").schema([S.OfStr])]).uniques([["defn", "name"], ["clause"]]),
    ])]))]);
}

const store: any = new B.Store();
const [Text, Int, Bool] = [store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()];
store.transaction(wts);
const [Definition, Clause] = [store.registered("wts.Definition"), store.registered("wts.Clause")];

## PAT-01 · A pattern: symbols, variables and constraints, in order, each its own; validated

In [ ]:
/** wts.Edit, a pattern over what `wts` installed. */
function edits(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("edit/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Edit: S.OfPattern })
    .spec([(_: any) => (
      _.Edit
        .name("wts.Edit")
        .description("A definition and one of its clauses")
        .symbols([
          (t: any) => t.name("defn").schema([txn.wts.Definition]),
          (t: any) => t.name("clause").schema([txn.wts.Clause]).description("C-1"),
        ])
        .variables([(t: any) => t.name("title").schema([S.OfStr]).description("its new title")])
        .requires([(c: any) => c.description("the title is new")]) // a constraint: its result, OfBool, by default
    )])]))]);
}

store.transaction(edits);
const Edit = store.registered("wts.Edit");
assert(Edit.validate().length === 0 && same([...Edit.symbols.keys()], ["defn", "clause"]) && same([...Edit.variables.keys()], ["title"]));
assert(Edit.variables.get("title").schema === Text && Edit.symbols.get("clause").description === "C-1");
assert(Edit.requires.length === 1 && Edit.requires[0].result === Bool); // a constraint: an expression of the store's OfBool
assert(store.registered("wts.Edit") === Edit && store.module("wts").schemas.get("Edit") === Edit); // named: in its module
store.transaction(() => {
  {
    const twin = new B.OfPattern.Builder(Edit, true, store).name("wts.Twin").clone();
    assert(twin.symbols.get("defn") !== Edit.symbols.get("defn") && twin.requires[0] !== Edit.requires[0]); // a clone owns its own
    assert(twin.variables.get("title") !== Edit.variables.get("title") && twin.variables.get("title").schema === Text);
  }
});
const [Port] = store.transaction(() => {
  const Port = store.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("pin").schema([Text])]));
  return [Port];
});
store.transaction(() => {
  {
    const bad = store.Schemas.OfPattern((p: any) => (
      p
        .symbols([(t: any) => t.name("x").schema([Port]), (t: any) => t.name("v")])
        .variables([(t: any) => t.name("v").schema([Definition]), (t: any) => t.name("w")])
    ));
    bad.variables.get("w").description = 1n; // what the builders refuse
    assert(same(bad.validate(), [
      "names used as both symbol and variable: ['v']",
      "symbol 'x': a symbol stands for a reference object, so its schema is a reference object schema",
      "variable 'v': a reference object schema cannot be a property's schema",
      "variable 'w': a description is text, got int"]));
    assert(same(new S.OfPattern.Data({ symbols: new Map([["p", new S.OfSymbol.Data({ name: "p" })]]),
      variables: new Map([["p", new S.OfVariable.Data({ name: "p" })]]) }).validate(), ["names used as both symbol and variable: ['p']"]));
  }
});

## PAT-02 · Expressions and constraints: a pattern and a result schema; a constraint's result is OfBool; a pattern requires its own constraints

In [ ]:
/** wts.Count, an expression over wts.Edit. */
function counts(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("count/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Count: S.OfExpression })
    .spec([(_: any) => _.Count.name("wts.Count").pattern([txn.wts.Edit]).result([S.OfInt])])]))]);
}

store.transaction(counts);
const Count = store.registered("wts.Count");
assert(Count.validate().length === 0 && Count.pattern === Edit && Count.result === Int);
assert(store.registered("wts.Count") === Count && Count.equals(Count) && !Count.equals(Edit));
store.transaction(() => {
  {
    const Holds = store.Schemas.OfConstraint((c: any) => c.name("wts.Holds").pattern([Edit])); // a builder's kind: its result fixed
    assert(Holds.result === Bool && Holds.validate().length === 0 && Holds instanceof S.OfExpression.Data);
    assert(new B.OfConstraint.Builder().create().result.name === "Schemas.OfBool"); // outside a store, the template's
    assert(same(new S.OfExpression.Data({ pattern: Count }).validate(), ["an expression's pattern is a pattern",
      "an expression needs a result schema"]));
    assert(same(new S.OfExpression.Data({ result: Port, name: "x y" }).validate(), [
      "name 'x y' is not identifiers separated by dots, e.g. 'crm.Contact'"]));
    assert(same(new S.OfExpression.Data({ result: Definition }).validate(), [
      "result: a reference object schema cannot be a property's schema"]));
    const own = new S.OfExpression.Data({ result: Bool });
    const mine = new S.OfPattern.Data({ requires: [own] });
    own.pattern = mine;
    assert(mine.validate().length === 0); // over its own pattern
    assert(same(new S.OfPattern.Data({ requires: [Count] }).validate(), [
      "requires[0] is a constraint: an expression whose result is Schemas.OfBool"]));
    assert(same(new S.OfPattern.Data({ requires: [Holds] }).validate(), ["requires[0] is over this pattern's frames, not another's"]));
    assert(same(new S.OfPattern.Data({ requires: [new S.OfExpression.Data({ result: Bool, description: 2n as never })] }).validate(), [
      "requires[0]: a description is text, got int"]));
  }
});

## PAT-03 · Collectors: a pattern, the names it collects, commutative or associative, two suffixes, and what it computes

In [ ]:
{
  /** wts.Sum, a collector of wts.Edit's frames, computing wts.Count. */
  const sums = (txn: any): any => txn.uid("sum/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Sum: txn.Schemas.OfCollector })
    .spec([(_: any) => (
      _.Sum
        .name("wts.Sum")
        .pattern([txn.wts.Edit])
        .symbols(["clause"])
        .variables(["title"])
        .commutative(true)
        .associative(true)
        .suffixes({ current: "_a", next: "_b" })
        .computes([txn.wts.Count])
    )])]))]);
  store.transaction(sums);
  const Sum = store.registered("wts.Sum");
  store.transaction(() => {
    assert(Sum.validate().length === 0 && same(Sum.symbols, ["clause"]) && same(Sum.variables, ["title"])
      && equal(Sum.suffixes, map({ current: "_a", next: "_b" })));
    assert(same(new B.OfCollector.Builder(Sum, false, store).symbols._held(), ["clause"]) && Sum.computes === Count && Sum.equals(Sum));
    const cleared = new B.OfCollector.Builder(Sum, true, store).name("wts.Cleared").symbols(null).variables(null).suffixes(null).clone();
    assert(cleared.symbols.length === 0 && cleared.variables.length === 0 && cleared.suffixes === null);
    const C = (fields: Record<string, unknown>) => new S.OfCollector.Data(fields as never).validate();
    assert(same(C({}), [
      "a collector needs a pattern",
      "a collector is commutative or associative, so that its result does not depend on the order frames arrive in"]));
    assert(same(C({ pattern: Edit, symbols: ["nope"], variables: ["none"], associative: true }), [
      "it collects symbols ['nope'], which its pattern does not declare",
      "it collects variables ['none'], which its pattern does not declare"]));
    assert(same(C({ pattern: Edit, commutative: true, suffixes: map({ current: "x" }) }), [
      "a collector's suffixes are 'current' and 'next'"]));
    assert(same(C({ pattern: Edit, commutative: true, suffixes: map({ current: "x", next: "x" }) }), [
      "a collector's 'current' and 'next' suffixes differ"]));
    assert(same(C({ pattern: Edit, commutative: true, computes: Edit }), ["what a collector computes is an expression"]));
    assert(same(C({ pattern: Edit, commutative: true, computes: new S.OfExpression.Data() }), [
      "computes: an expression needs a result schema"]));
    assert(new B.OfCollector.Builder(store.Schemas.OfCollector((c: any) => c), false, store).suffixes._held() === null);
  });
}

## PAT-04 · Patterns, expressions and collectors in a store's snapshot: written once, read back the same

In [ ]:
{
  const log = Plain.ToPlain(store).Snapshot() as any[];
  const back: any = new B.Store();
  Plain.FromPlain(back).Snapshot(log);
  assert(Json.dumps(Plain.ToPlain(back).Snapshot() as never) === Json.dumps(log as never));
  assert(Json.dumps(Json.loads(Json.ToJSON(back).Snapshot())) === Json.dumps(log as never));
  const read = back.registered("wts.Edit");
  assert(read !== Edit && same([...read.symbols.keys()], ["defn", "clause"]) && read.requires[0].result === back.Schemas.OfBool.as_schema());
  assert(back.registered("wts.Sum").pattern === read && back.registered("wts.Sum").computes === back.registered("wts.Count"));
}